# 🏛️ Indian Legal LLM — LegalDelta with IL-TUR + Bail Data

**Paper:** [LegalDelta (ICASSP 2026)](https://arxiv.org/abs/2508.12281): Information Gain Reward for Legal Reasoning  
**Datasets:**  
- [IL-TUR](https://huggingface.co/datasets/Exploration-Lab/IL-TUR) (ACL 2024) — 34K Indian SC cases, judgment prediction + explanation  
- [IndianBailJudgments-1200](https://huggingface.co/datasets/SnehaDeshmukh/IndianBailJudgments-1200) — 1.2K annotated bail orders with IPC sections  

**Model:** Qwen2.5-1.5B-Instruct (4-bit, LoRA)  
**API:** OpenRouter (configurable models)

| Stage | What | GPU? |
|-------|------|------|
| 1 | Install + Clone + Load Data | ❌ |
| 2 | Explore Datasets | ❌ |
| 3 | Generate Dual-Mode CoT Pairs (API) | ❌ |
| 4 | Load Model + Test IG Reward | ✅ |
| 5 | GRPO Training | ✅ |
| 6 | Evaluation + Ablation | ✅ |

---

# ═══════════════════════════════════════
# STAGE 1 — Setup (No GPU)
# ═══════════════════════════════════════

In [ ]:
# 1.1 Install Dependencies
!pip install -q transformers datasets peft trl accelerate bitsandbytes
!pip install -q sentencepiece protobuf huggingface_hub
!pip install -q rouge-score bert-score nltk evaluate
!pip install -q openai pandas numpy tqdm psutil

In [ ]:
# 1.2 Clone Repo + Setup Paths
import sys, os, gc, json
import numpy as np
import pandas as pd
from pathlib import Path

# ╔══════════════════════════════════════════════════╗
# ║  ⚙️ CONFIG — Change these as needed              ║
# ╚══════════════════════════════════════════════════╝
REPO_URL  = "https://github.com/highintoxic/RLProject.git"
REPO_NAME = "RLProject"

IS_KAGGLE = os.path.exists('/kaggle')
IS_COLAB  = 'google.colab' in sys.modules
ENV = 'Kaggle' if IS_KAGGLE else ('Colab' if IS_COLAB else 'Local')
print(f"🖥️ Environment: {ENV}")

if IS_KAGGLE:
    WORK_DIR = f'/kaggle/working/{REPO_NAME}'
    CLONE_DIR = '/kaggle/working'
elif IS_COLAB:
    WORK_DIR = f'/content/{REPO_NAME}'
    CLONE_DIR = '/content'
else:
    WORK_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
    CLONE_DIR = None

import subprocess
if not os.path.isdir(WORK_DIR) and CLONE_DIR:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL], cwd=CLONE_DIR, check=True)

if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)
os.chdir(WORK_DIR)
print(f"📂 Working dir: {os.getcwd()}")
print(f"📦 src/ found: {os.path.exists('src/__init__.py')}")

In [ ]:
# 1.3 API Keys
from src.config import HF_TOKEN, OPENROUTER_API_KEY
from huggingface_hub import login

if HF_TOKEN:
    login(HF_TOKEN)
    print("✅ HuggingFace authenticated")
else:
    print("⚠️ HF_TOKEN not set — IL-TUR is gated, you need this!")

print(f"OpenRouter API: {'✅' if OPENROUTER_API_KEY else '❌ Missing'}")

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  ⚙️ MODEL SELECTION — Change models here         ║
# ╚══════════════════════════════════════════════════╝
import src.config as cfg

# Reasoner model (for CoT generation — needs reasoning capability)
cfg.REASONER_MODEL = "deepseek/deepseek-r1:free"

# Chat model (for direct answers / LLM-as-judge — any good chat model)
cfg.CHAT_MODEL = "deepseek/deepseek-chat-v3-0324:free"

# Browse all models: https://openrouter.ai/models (filter by 'free' for zero cost)
# Other good options:
#   cfg.REASONER_MODEL = "google/gemini-2.5-flash:free"
#   cfg.CHAT_MODEL     = "meta-llama/llama-4-maverick:free"

print(f"Reasoner: {cfg.REASONER_MODEL}")
print(f"Chat:     {cfg.CHAT_MODEL}")

# ═══════════════════════════════════════
# STAGE 2 — Load & Explore Datasets (No GPU)
# ═══════════════════════════════════════

In [ ]:
# 2.1 Load IL-TUR CJPE (Court Judgment Prediction & Explanation)
from datasets import load_dataset

# IL-TUR is gated — you must accept terms at:
# https://huggingface.co/datasets/Exploration-Lab/IL-TUR

# Load each CJPE split from the parquet subdirectory
cjpe = {}
for split in ['single_train', 'single_dev', 'multi_train', 'multi_dev', 'test', 'expert']:
    try:
        ds = load_dataset(
            'Exploration-Lab/IL-TUR',
            data_files=f'cjpe/{split}*.parquet',
            split='train',
        )
        cjpe[split] = ds
        print(f'  {split}: {len(ds)} samples')
    except Exception as e:
        print(f'  {split}: skipped ({e})')

print(f'\nLoaded {len(cjpe)} splits')
if 'single_train' in cjpe:
    print(f'Columns: {cjpe["single_train"].column_names}')

In [ ]:
# 2.2 Preview a CJPE sample
sample = cjpe['single_train'][0]
print(f"ID: {sample['id']}")
print(f"Label: {sample['label']} (0=rejected, 1=accepted)")
print(f"Text (first 500 chars):")
print(sample['text'][:500])

In [ ]:
# 2.3 Load IndianBailJudgments-1200 (supplementary)
bail = load_dataset("SnehaDeshmukh/IndianBailJudgments-1200", split="train")

print(f"\n=== IndianBailJudgments ===")
print(f"  Total: {len(bail)} samples")
print(f"  Columns: {bail.column_names}")
print(f"\n  Sample fields:")
for k, v in bail[0].items():
    print(f"    {k}: {str(v)[:200]}")

In [ ]:
# 2.4 Build unified training set from CJPE
# Use single_train (5K single-appeal cases) as main training data
train_data = cjpe['single_train']
test_data  = cjpe['test']

# Label distribution
labels = train_data['label']
n_accepted = sum(1 for l in labels if l == 1)
n_rejected = sum(1 for l in labels if l == 0)
print(f"Training set: {len(train_data)} cases")
print(f"  Accepted: {n_accepted} ({100*n_accepted/len(train_data):.1f}%)")
print(f"  Rejected: {n_rejected} ({100*n_rejected/len(train_data):.1f}%)")
print(f"Test set: {len(test_data)} cases")

print("\n✅ Stage 2 complete — datasets loaded!")

# ═══════════════════════════════════════
# STAGE 3 — Generate Dual-Mode Pairs (No GPU)
# ═══════════════════════════════════════

> Uses OpenRouter API. Cost: ~$1–2 for 300 cases.

In [ ]:
DUAL_MODE_PATH = "dual_mode_iltur.jsonl"
N_DUAL_SAMPLES = 300

if os.path.exists(DUAL_MODE_PATH):
    with open(DUAL_MODE_PATH) as f:
        existing = len(f.readlines())
    print(f"⏩ CHECKPOINT: {DUAL_MODE_PATH} ({existing} pairs)")
    SKIP_GENERATION = True
else:
    print(f"Will generate {N_DUAL_SAMPLES} dual-mode pairs.")
    SKIP_GENERATION = False

In [ ]:
# 3.1 Test single dual-mode pair
if not SKIP_GENERATION:
    from src.legaldelta import create_openrouter_client, generate_dual_mode_pair
    
    client = create_openrouter_client()
    
    # IL-TUR CJPE has 'text' field with full case document
    test_facts = train_data[0]['text'][:1000]
    
    pair = generate_dual_mode_pair(
        client, test_facts,
        reasoner_model=cfg.REASONER_MODEL,
        chat_model=cfg.CHAT_MODEL,
    )
    
    print("=== DIRECT ANSWER ===")
    print((pair['direct_answer'] or '(empty)')[:400])
    print("\n=== CoT REASONING ===")
    print((pair['cot_reasoning'] or '(empty)')[:400])
    print("\n=== CoT ANSWER ===")
    print((pair['cot_answer'] or '(empty)')[:400])
else:
    print("⏩ Skipped — using checkpoint.")

In [ ]:
# 3.2 Batch generate dual-mode pairs
if not SKIP_GENERATION:
    dual_data = []
    import json
    
    for i in range(N_DUAL_SAMPLES):
        case = train_data[i]
        facts = case['text'][:1000]
        try:
            pair = generate_dual_mode_pair(
                client, facts,
                reasoner_model=cfg.REASONER_MODEL,
                chat_model=cfg.CHAT_MODEL,
            )
            pair['label'] = case['label']
            pair['case_id'] = case.get('id', str(i))
            dual_data.append(pair)
            
            if (i + 1) % 50 == 0:
                with open(DUAL_MODE_PATH, 'w', encoding='utf-8') as f:
                    for row in dual_data:
                        f.write(json.dumps(row, ensure_ascii=False) + '\n')
                print(f"  💾 Saved {i+1}/{N_DUAL_SAMPLES}")
        except Exception as e:
            print(f"  ⚠️ Skipped {i}: {e}")
    
    # Final save
    with open(DUAL_MODE_PATH, 'w', encoding='utf-8') as f:
        for row in dual_data:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
    print(f"\n✅ Generated {len(dual_data)} pairs → {DUAL_MODE_PATH}")
else:
    print("⏩ Skipped.")

In [ ]:
# 3.3 Load dual-mode data
import json

with open(DUAL_MODE_PATH, 'r', encoding='utf-8') as f:
    dual_data = [json.loads(line) for line in f]

print(f"✅ Loaded {len(dual_data)} dual-mode pairs")
direct_lens = [len(d.get('direct_answer','')) for d in dual_data]
cot_lens = [len(d.get('cot_answer','')) for d in dual_data]
print(f"  Avg direct answer: {np.mean(direct_lens):.0f} chars")
print(f"  Avg CoT answer:    {np.mean(cot_lens):.0f} chars")

print("\n✅ Stage 3 complete — no GPU used!")

# ═══════════════════════════════════════
# STAGE 4 — Load Model + Test IG Reward (GPU)
# ═══════════════════════════════════════

In [ ]:
# 4.1 GPU check
import torch

if not torch.cuda.is_available():
    print("❌ No GPU. Enable: Settings → Accelerator → GPU T4")
    print("   Stages 1-3 are done. Restart with GPU for Stage 4+.")
else:
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {gpu} ({vram:.1f} GB)")

In [ ]:
# 4.2 Load base model
from src.model_utils import load_base_model

model, tokenizer = load_base_model()

In [ ]:
# 4.3 Test Information Gain on a sample pair
from src.legaldelta import compute_information_gain, compute_legaldelta_reward

s = dual_data[0]
ig = compute_information_gain(
    model, tokenizer,
    facts=s.get('facts', ''),
    direct_answer=s.get('direct_answer', ''),
    cot_response=s.get('cot_answer', ''),
)
reward = compute_legaldelta_reward(
    facts=s.get('facts', ''),
    direct_answer=s.get('direct_answer', ''),
    cot_full_response=s.get('cot_answer', ''),
    reference=str(s.get('label', '')),
    ig_score=ig,
)

print(f"=== Information Gain Test ===")
print(f"  IG score:    {ig:.4f} (>0.1 = CoT helps)")
print(f"  Total reward: {reward['total']:.4f}")
print(f"  Structure:   {reward['structure']:.4f}")
print(f"  Domain:      {reward['domain']:.4f}")

In [ ]:
# 4.4 IG across 10 samples
ig_scores = []
for i in range(min(10, len(dual_data))):
    s = dual_data[i]
    ig = compute_information_gain(
        model, tokenizer,
        s.get('facts',''), s.get('direct_answer',''), s.get('cot_answer',''),
    )
    ig_scores.append(ig)
    print(f"  Sample {i}: IG={ig:.4f}")

print(f"\nMean IG: {np.mean(ig_scores):.4f}")
print("✅ Stage 4 complete — IG reward works!")

# ═══════════════════════════════════════
# STAGE 5 — GRPO Training (GPU)
# ═══════════════════════════════════════

In [ ]:
SAVE_DIR = "./legaldelta-iltur-1.5b"
TRAINING_DONE = os.path.exists(os.path.join(SAVE_DIR, "adapter_config.json"))

if TRAINING_DONE:
    print(f"⏩ CHECKPOINT: {SAVE_DIR} exists")
else:
    print("Will train with LegalDelta reward.")

In [ ]:
# 5.1 Apply LoRA
if not TRAINING_DONE:
    from src.model_utils import apply_lora
    model, lora_config = apply_lora(model)

In [ ]:
# 5.2 Format for GRPO
if not TRAINING_DONE:
    from datasets import Dataset
    from src.config import INFERENCE_SYSTEM_PROMPT
    
    def format_grpo(row):
        facts = row.get('facts', '')[:800]
        return {
            "prompt": (
                f"<|im_start|>system\n{INFERENCE_SYSTEM_PROMPT}<|im_end|>\n"
                f"<|im_start|>user\n"
                f"Predict the judgment for this case and explain your reasoning "
                f"with applicable IPC/BNS sections:\n\n{facts}\n"
                f"<|im_end|>\n<|im_start|>assistant\n"
            ),
        }
    
    grpo_ds = Dataset.from_list([format_grpo(r) for r in dual_data])
    print(f"GRPO dataset: {len(grpo_ds)} samples")

In [ ]:
# 5.3 Define reward functions
if not TRAINING_DONE:
    from src.reward_functions import (
        reward_has_legal_citation,
        reward_has_reasoning,
        reward_bilingual,
    )
    from src.legaldelta import compute_legaldelta_reward
    
    def legaldelta_reward(completions, **kwargs):
        rewards = []
        for c in completions:
            r = compute_legaldelta_reward(
                facts='', direct_answer='',
                cot_full_response=c, reference='',
                ig_score=0.3,  # fixed during training for speed
            )
            rewards.append(r['total'])
        return rewards
    
    ALL_REWARDS = [legaldelta_reward, reward_has_legal_citation, reward_has_reasoning, reward_bilingual]
    print(f"Reward functions: {len(ALL_REWARDS)}")

In [ ]:
# 5.4 Train
if not TRAINING_DONE:
    from trl import GRPOTrainer, GRPOConfig
    
    grpo_config = GRPOConfig(
        output_dir="./grpo-checkpoints",
        learning_rate=5e-6,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,
        num_generations=4,
        max_new_tokens=512,
        num_train_epochs=1,
        fp16=True,
        report_to="none",
        save_steps=100,
    )
    
    trainer = GRPOTrainer(
        model=model,
        tokenizer=tokenizer,
        reward_funcs=ALL_REWARDS,
        args=grpo_config,
        train_dataset=grpo_ds,
    )
    
    print("🚀 Training...")
    trainer.train()
    trainer.save_model(SAVE_DIR)
    print(f"✅ Saved to {SAVE_DIR}")
    
    del trainer
    gc.collect()
    torch.cuda.empty_cache()
else:
    print("⏩ Skipped.")

# ═══════════════════════════════════════
# STAGE 6 — Evaluation + Ablation (GPU)
# ═══════════════════════════════════════

In [ ]:
# 6.1 Load models for comparison
from src.model_utils import load_base_model, load_finetuned_model, generate_judgment

if 'model' in dir():
    del model
gc.collect()
torch.cuda.empty_cache()

base_model, tokenizer = load_base_model()

if os.path.exists(SAVE_DIR):
    ft_model, _ = load_finetuned_model(SAVE_DIR)
else:
    ft_model = None
    print("⚠️ No trained model — eval on base only")

In [ ]:
# 6.2 LegalDelta Ablation Table
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
n_eval = min(50, len(test_data))

base_scores, ft_scores = [], []

for i in range(n_eval):
    case = test_data[i]
    facts = case['text'][:800]
    ref = 'accepted' if case['label'] == 1 else 'rejected'
    
    base_out = generate_judgment(base_model, tokenizer, facts, max_new_tokens=256)
    base_scores.append(scorer.score(ref, base_out)['rougeL'].fmeasure)
    
    if ft_model:
        ft_out = generate_judgment(ft_model, tokenizer, facts, max_new_tokens=512)
        ft_scores.append(scorer.score(ref, ft_out)['rougeL'].fmeasure)
    
    if (i+1) % 10 == 0:
        print(f"  Done {i+1}/{n_eval}")

print(f"\n{'='*55}")
print(f"  LegalDelta Ablation — IL-TUR CJPE")
print(f"{'='*55}")
print(f"{'Mode':<30} {'ROUGE-L':>10} {'Delta':>10}")
print(f"{'-'*55}")
base_avg = np.mean(base_scores)
print(f"{'Base (Qwen 1.5B)':<30} {base_avg:>10.4f} {'baseline':>10}")
if ft_scores:
    ft_avg = np.mean(ft_scores)
    print(f"{'+ LegalDelta GRPO':<30} {ft_avg:>10.4f} {ft_avg-base_avg:>+10.4f}")
print(f"{'='*55}")

In [ ]:
# 6.3 Side-by-side examples
for idx in [0, 10, 25]:
    if idx >= len(test_data):
        break
    case = test_data[idx]
    facts = case['text'][:600]
    label = 'ACCEPTED' if case['label'] == 1 else 'REJECTED'
    
    print(f"\n{'='*60}")
    print(f"CASE {idx} — Ground truth: {label}")
    print(f"{'='*60}")
    print(f"Facts: {facts[:200]}...\n")
    
    base_out = generate_judgment(base_model, tokenizer, facts, max_new_tokens=256)
    print(f"--- BASE MODEL ---")
    print(base_out[:400])
    
    if ft_model:
        ft_out = generate_judgment(ft_model, tokenizer, facts, max_new_tokens=512)
        print(f"\n--- LEGALDELTA MODEL ---")
        print(ft_out[:400])

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  🎉 COMPLETE                                            ║
# ║                                                          ║
# ║  Paper:   LegalDelta (ICASSP 2026)                      ║
# ║  Dataset: IL-TUR CJPE (ACL 2024) — 34K SC cases         ║
# ║  Model:   Qwen2.5-1.5B + LoRA + GRPO                   ║
# ║  Reward:  Information Gain + Structure + Domain          ║
# ║                                                          ║
# ║  Our contribution:                                       ║
# ║  → LegalDelta adapted from Chinese → Indian legal       ║
# ║  → IL-TUR benchmark for evaluation                      ║
# ║  → 14B → 1.5B (resource efficient)                     ║
# ╚══════════════════════════════════════════════════════════╝
print("\n🎉 Pipeline complete!")